# Genre Based Model

In [2]:
#ingest downloaded csvs
import pandas as pd
movies = pd.read_csv('movies.csv')
principals  = pd.read_csv('principals.csv')
ratings = pd.read_csv('ratings.csv')
plotline = pd.read_csv('plotline.csv')
names = pd.read_csv('names.csv')

### 1. Vectorization

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

#-------------------- sanjana's change ---------------------
#cleaned up Genres Column for the CountVectorizer
movies['Genre'] = movies['Genre'].fillna('')
movies['Genre'] = movies['Genre'].str.replace(' ', '')
movies['Genre'] = movies['Genre'].str.lower()
#-------------------- sanjana's change ---------------------


#Change Genre list to pipe-separated string for CountVectorizer
movies['Genre'] = movies['Genre'].apply(lambda x: '|'.join(x) if isinstance(x, list) else x)  

#simplifying the tokenization by using the pipe as a separator and setting token_pattern to None

#-------------------- sanjana's change ---------------------

count_vectorizer = CountVectorizer(tokenizer = lambda x: x.split('|'), token_pattern = None, binary = True)

genre_matrix = count_vectorizer.fit_transform(movies['Genre'])
#-------------------- sanjana's change ---------------------


genre_features = pd.DataFrame(
    genre_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=movies['Title']
)

print(f'Genres list: {list(count_vectorizer.get_feature_names_out())}')
genre_features.head()

Genres list: ["['action','adventure','animation']", "['action','adventure','biography']", "['action','adventure','comedy']", "['action','adventure','crime']", "['action','adventure','drama']", "['action','adventure','family']", "['action','adventure','fantasy']", "['action','adventure','history']", "['action','adventure','horror']", "['action','adventure','mystery']", "['action','adventure','romance']", "['action','adventure','sci-fi']", "['action','adventure','thriller']", "['action','adventure','war']", "['action','adventure']", "['action','animation','comedy']", "['action','animation','crime']", "['action','animation','drama']", "['action','animation','family']", "['action','animation','fantasy']", "['action','animation','horror']", "['action','animation','music']", "['action','animation','sci-fi']", "['action','biography','comedy']", "['action','biography','crime']", "['action','biography','drama']", "['action','biography','history']", "['action','comedy','crime']", "['action','com

,"['action','adventure','animation']","['action','adventure','biography']","['action','adventure','comedy']","['action','adventure','crime']","['action','adventure','drama']","['action','adventure','family']","['action','adventure','fantasy']","['action','adventure','history']","['action','adventure','horror']","['action','adventure','mystery']",...,"['romance','thriller','war']","['romance','thriller']","['romance','war']","['romance','western']",['romance'],"['sci-fi','thriller']",['sci-fi'],"['thriller','war']",['thriller'],['western']
Title,,,,,,,,,,,,,,,,,,,,,
Dante's Inferno (L'inferno),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
The Student of Prague (Der Student von Prag),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
The Last Days of Pompeii (Gli ultimi giorni di Pompei),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Julius Caesar (Cajus Julius Caesar),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
"His Majesty, the Scarecrow of Oz",0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2. Compute cosine similarity

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(genre_matrix)

similarity_df = pd.DataFrame(similarity, index=movies['Title'], columns=movies['Title'])

similarity_df.iloc[:5, :5]

Title,Dante's Inferno (L'inferno),The Student of Prague (Der Student von Prag),The Last Days of Pompeii (Gli ultimi giorni di Pompei),Julius Caesar (Cajus Julius Caesar),"His Majesty, the Scarecrow of Oz"
Title,,,,,
Dante's Inferno (L'inferno),1.0,0.0,0.0,0.0,0.0
The Student of Prague (Der Student von Prag),0.0,1.0,0.0,0.0,0.0
The Last Days of Pompeii (Gli ultimi giorni di Pompei),0.0,0.0,1.0,0.0,0.0
Julius Caesar (Cajus Julius Caesar),0.0,0.0,0.0,1.0,0.0
"His Majesty, the Scarecrow of Oz",0.0,0.0,0.0,0.0,1.0


### 3. genre recommendation function

In [5]:

def genre_recommend(title, n=10):
    
    # Check if title exists in similarity_df
    if title not in similarity_df.columns:
        print(f'Title not found: "{title}"')
        matches = [t for t in similarity_df.columns if title.lower() in t.lower()]
        print('Possible matches:', matches[:10])
        return None

    # for duplicate titles
    count = list(similarity_df.columns).count(title)
    if count > 1:
        # lookup row - the best one by rating
        best_idx = movies[movies['Title'] == title].sort_values('Weighted Rating', ascending=False).index[0]
        # movies row = similarity_df column position
        col_pos = movies.index.get_loc(best_idx)
        best = movies.loc[best_idx]
        print(f'Auto-selected: {best["Title"]} ({best["Release Year"]}) — Rating: {best["Weighted Rating"]}')
    else:
        col_pos = list(similarity_df.columns).index(title)

    #  similarity score 
    sim_df = pd.DataFrame({
        'Title': list(similarity_df.index),
        'Similarity': similarity_df.iloc[:, col_pos].values
    })

    # Exclude the input title and get top N similar movies
    top_n = sim_df[sim_df['Title'] != title].sort_values('Similarity', ascending=False).head(n)

    # Build results with movie metadata
    result = top_n.merge(
        movies[['Movie ID', 'Title', 'Genre', 'Release Year', 'Weighted Rating']],
        on='Title'
    )
    result = result.rename(columns={'Similarity': 'Genre Similarity Score'})
    result['Genre Similarity Score'] = result['Genre Similarity Score'].round(4)

    return result.sort_values('Weighted Rating', ascending=False).reset_index(drop=True)

## Test cases

In [6]:
# Test 1: The Dark Knight
input1 = 'The Dark Knight'

print(f'Genre Recommendations for: {input1}')
recommend1 = genre_recommend(input1, n=10)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating', 'Genre Similarity Score']]

Genre Recommendations for: The Dark Knight


,Movie ID,Title,Genre,Weighted Rating,Genre Similarity Score
0,tt0220757,Shool,"['action','crime','drama']",7.7,1.0
1,tt0083190,Thief,"['action','crime','drama']",7.4,1.0
2,tt2140479,The Accountant,"['action','crime','drama']",7.3,1.0
3,tt0107206,In the Line of Fire,"['action','crime','drama']",7.2,1.0
4,tt2568760,Theevram,"['action','crime','drama']",6.4,1.0
5,tt0113501,Just Cause,"['action','crime','drama']",6.4,1.0
6,tt0091428,Legacy of Rage (Long zai jiang hu),"['action','crime','drama']",6.1,1.0
7,tt0102217,The King of the Kickboxers,"['action','crime','drama']",5.9,1.0
8,tt2140037,Jane Got a Gun,"['action','crime','drama']",5.9,1.0
9,tt0091604,Naked Vengeance,"['action','crime','drama']",5.8,1.0


In [7]:
# Test 2: A Nightmare on Elm Street
input2 = 'A Nightmare on Elm Street'

print(f' Genre Recommendations for: {input2}')
recommend2 = genre_recommend(input2, n=10)
recommend2[['Movie ID', 'Title', 'Genre', 'Weighted Rating', 'Genre Similarity Score']]

 Genre Recommendations for: A Nightmare on Elm Street
Auto-selected: A Nightmare on Elm Street (1984) — Rating: 7.4


,Movie ID,Title,Genre,Weighted Rating,Genre Similarity Score
0,tt0082235,Dark Night of the Scarecrow,['horror'],6.7,1.0
1,tt0070770,Tales That Witness Madness,['horror'],5.7,1.0
2,tt0076853,The Uncanny,['horror'],5.7,1.0
3,tt0083629,The Beast Within,['horror'],5.6,1.0
4,tt0070673,The Severed Arm,['horror'],5.1,1.0
5,tt0079855,Savage Weekend,['horror'],4.6,1.0
6,tt0091990,Sorority House Massacre,['horror'],4.5,1.0
7,tt0096808,Amityville Horror: The Evil Escapes,['horror'],4.5,1.0
8,tt0082237,Dawn of the Mummy,['horror'],4.1,1.0
9,tt0410332,August Underground's Mordum,['horror'],3.1,1.0


In [8]:
# Test 3: 12 Angry Men
input3 = '12 Angry Men'

print(f' Genre Recommendations for: {input3}')
recommend3 = genre_recommend(input3, n=10)
recommend3[['Movie ID', 'Title', 'Genre', 'Weighted Rating', 'Genre Similarity Score']]

 Genre Recommendations for: 12 Angry Men
Auto-selected: 12 Angry Men (1957) — Rating: 9.0


,Movie ID,Title,Genre,Weighted Rating,Genre Similarity Score
0,tt0075314,Taxi Driver,"['crime','drama']",8.2,1.0
1,tt0112641,Casino,"['crime','drama']",8.2,1.0
2,tt0110729,Once Were Warriors,"['crime','drama']",7.9,1.0
3,tt0055618,Whistle Down the Wind,"['crime','drama']",7.6,1.0
4,tt1916728,Shor in the City,"['crime','drama']",7.2,1.0
5,tt0039947,The Upturned Glass,"['crime','drama']",6.9,1.0
6,tt0068341,The Valachi Papers,"['crime','drama']",6.4,1.0
7,tt0102989,Straight Out of Brooklyn,"['crime','drama']",5.9,1.0
8,tt0305973,Sonny,"['crime','drama']",5.6,1.0
9,tt0066286,The Rebel Rousers,"['crime','drama']",4.4,1.0


# NLP Model

In [9]:
# Import TfidfVectorizer from sklearn.feature_extraction.text
from sklearn.feature_extraction.text import TfidfVectorizer

# Import cosine_similarity from sklearn.metrics.pairwise
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

Libraries imported successfully.


Now, let's initialize the `TfidfVectorizer`. We'll use `stop_words='english'` to remove common English words that don't add much meaning for similarity calculation. We will also fill any potential NaN values in the 'Plot Synopsis' with an empty string to prevent errors.

In [10]:
# Initialize TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Fill NaN values in 'Plot Synopsis' with an empty string to avoid errors
movies['Plot Synopsis'] = movies['Plot Synopsis'].fillna('')

# Construct the TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(movies['Plot Synopsis'])

# Print the shape of the TF-IDF matrix
print(f"Shape of TF-IDF matrix: {tfidf_matrix.shape}")

Shape of TF-IDF matrix: (12857, 196462)


Next, we will compute the cosine similarity matrix. This matrix will give us a score indicating the similarity between each movie's plot synopsis.

In [11]:
# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Print the shape of the cosine similarity matrix
print(f"Shape of cosine similarity matrix: {cosine_sim.shape}")

Shape of cosine similarity matrix: (12857, 12857)


attempt 3: uses combo of genre and plot synopsis

In [12]:
import re
import ast
import numpy as np
import pandas as pd

#Cleans and standardizes movie titles so you can compare them reliably
def normalize_title(title):
    title = str(title).lower().strip()
    title = re.sub(r'\(\d{4}\)', '', title)      # remove years like (1997)
    title = re.sub(r'[^a-z0-9\s]', ' ', title)   # remove punctuation
    title = re.sub(r'\s+', ' ', title).strip()   # clean extra spaces
    return title

#Checks if two movies are basically the same franchise / remake / sequel
def is_related_title(input_title, candidate_title):
    input_norm = normalize_title(input_title)
    cand_norm = normalize_title(candidate_title)

    if cand_norm == input_norm:
        return True

    # remove titles that contain the input title as a phrase
    pattern = rf'\b{re.escape(input_norm)}\b'
    if re.search(pattern, cand_norm):
        return True

    return False

#Turns your genre column into a clean Python set.
#Converts genre values like: "['Family', 'Fantasy', 'Musical'] into a Python set: {'Family', 'Fantasy', 'Musical'}
def parse_genres(genre_value):
    if pd.isna(genre_value):
        return set()

    if isinstance(genre_value, list):
        return set([str(g).strip() for g in genre_value if str(g).strip()])

    genre_value = str(genre_value).strip()

    # try to parse string representation of list
    try:
        parsed = ast.literal_eval(genre_value)
        if isinstance(parsed, list):
            return set([str(g).strip() for g in parsed if str(g).strip()])
    except:
        pass

    # fallback: split on commas
    return set([g.strip() for g in genre_value.split(',') if g.strip()])

#Measures how similar two movies are based on genre.
def genre_overlap_score(input_genres, candidate_genres):
    """
    Jaccard overlap = intersection / union
    """
    union = input_genres | candidate_genres
    if len(union) == 0:
        return 0.0
    return len(input_genres & candidate_genres) / len(union)


   
    Hybrid recommender:
    - plot similarity from TF-IDF cosine similarity
    - genre overlap for thematic relevance
    - removes sequel/remake/franchise-like titles

In [13]:
#attempt 3 movie recommender system
def get_recommendations_hybrid(movie_title, cosine_sim_matrix=cosine_sim, df=movies,
                               top_n=10, candidate_pool=50,
                               plot_weight=0.80, genre_weight=0.20):

#handles duplicate titles here:
    matching_movies = df[df['Title'] == movie_title].copy()

    if matching_movies.empty:
        return f"Movie Title not found: '{movie_title}'."

    if len(matching_movies) > 1:
        selected_movie_info = matching_movies.sort_values(
            'Weighted Rating', ascending=False
        ).iloc[0]

        matching_movies = df[
            (df['Title'] == movie_title) &
            (df['Release Year'] == selected_movie_info['Release Year'])
        ].iloc[[0]]

        print(
            f"Multiple movies found with the title '{movie_title}'. "
            f"Auto-selected: {selected_movie_info['Title']} "
            f"({int(selected_movie_info['Release Year'])}) — "
            f"Weighted Rating: {selected_movie_info['Weighted Rating']:.2f}"
        )

    input_movie = matching_movies.iloc[0]
    idx = matching_movies.index[0]

    # plot similarity
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # skip itself, keep a larger pool before filtering
    sim_scores = sim_scores[1:candidate_pool + 1]

    movie_indices = [i[0] for i in sim_scores]
    similarity_values = [i[1] for i in sim_scores]

    recommended_movies_df = df.iloc[movie_indices].copy()
    recommended_movies_df['Plot Similarity Score'] = similarity_values

    # genre overlap
    input_genres = parse_genres(input_movie['Genre'])
    recommended_movies_df['Genre Overlap Score'] = recommended_movies_df['Genre'].apply(
        lambda x: genre_overlap_score(input_genres, parse_genres(x))
    )

    # final weighted score
    recommended_movies_df['Final Score'] = (
        plot_weight * recommended_movies_df['Plot Similarity Score'] +
        genre_weight * recommended_movies_df['Genre Overlap Score']
    )

    # filter sequel/remake/franchise-like titles
    filtered_df = recommended_movies_df[
        ~recommended_movies_df['Title'].apply(
            lambda x: is_related_title(input_movie['Title'], x)
        )
    ].copy()

    # round for display
    filtered_df['Plot Similarity Score'] = filtered_df['Plot Similarity Score'].round(4)
    filtered_df['Genre Overlap Score'] = filtered_df['Genre Overlap Score'].round(4)
    filtered_df['Final Score'] = filtered_df['Final Score'].round(4)

    result = filtered_df[[
        'Movie ID', 'Title', 'Genre', 'Release Year',
        'Weighted Rating', 'Plot Similarity Score',
        'Genre Overlap Score', 'Final Score'
    ]]

    return result.sort_values(
        by=['Final Score', 'Plot Similarity Score', 'Weighted Rating'],
        ascending=[False, False, False]
    ).head(top_n).reset_index(drop=True)

In [14]:
# Test 1: The Dark Knight
input1 = 'The Dark Knight'

print(f'NLP Recommendations for: {input1}')
recommend1 = get_recommendations_hybrid(input1)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating']]

NLP Recommendations for: The Dark Knight


,Movie ID,Title,Genre,Weighted Rating
0,tt4853102,Batman: The Killing Joke,"['action','animation','crime']",6.4
1,tt0096895,Batman,"['action','adventure']",7.5
2,tt0372784,Batman Begins,"['action','crime','drama']",8.2
3,tt5973626,Batman: Return of the Caped Crusaders,"['action','adventure','animation']",6.7
4,tt0106364,Batman: Mask of the Phantasm,"['action','adventure','animation']",7.8
5,tt0112462,Batman Forever,"['action','adventure']",5.5
6,tt0051525,The Defiant Ones,"['crime','drama']",7.5
7,tt0093058,Full Metal Jacket,"['drama','war']",8.2
8,tt3139072,Son of Batman,"['action','adventure','animation']",6.7
9,tt0103776,Batman Returns,"['action','crime','fantasy']",7.1


In [15]:
# Test 2: A Nightmare on Elm Street
input1 = 'A Nightmare on Elm Street'

print(f'NLP Recommendations for: {input1}')
recommend1 = get_recommendations_hybrid(input1)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating']]

NLP Recommendations for: A Nightmare on Elm Street
Multiple movies found with the title 'A Nightmare on Elm Street'. Auto-selected: A Nightmare on Elm Street (1984) — Weighted Rating: 7.40


,Movie ID,Title,Genre,Weighted Rating
0,tt0050201,Blood of Dracula,['horror'],4.7
1,tt4052882,The Shallows,"['drama','horror','thriller']",6.3
2,tt3064298,Man Up,"['comedy','drama','romance']",6.8
3,tt0085159,Amityville 3-D,['horror'],4.2
4,tt0038700,The Locket,"['crime','drama','film-noir']",7.1
5,tt0329101,Freddy vs. Jason,['horror'],5.8
6,tt0065669,The Dunwich Horror,['horror'],5.4
7,tt0180274,Unhinged,['horror'],5.0
8,tt0100827,Two Evil Eyes (Due occhi diabolici),['horror'],6.1
9,tt0096808,Amityville Horror: The Evil Escapes,['horror'],4.5


In [16]:
# Test 3: Cinderella
input1 = 'Cinderella'

print(f'NLP Recommendations for: {input1}')
recommend1 = get_recommendations_hybrid(input1)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating']]

NLP Recommendations for: Cinderella
Multiple movies found with the title 'Cinderella'. Auto-selected: Cinderella (1950) — Weighted Rating: 7.30


,Movie ID,Title,Genre,Weighted Rating
0,tt2180411,Into the Woods,"['adventure','comedy','drama']",5.9
1,tt0160429,The King and I,"['animation','family','fantasy']",4.5
2,tt0048124,The Glass Slipper,"['family','fantasy','musical']",6.4
3,tt0053716,Cinderfella,"['comedy','family','fantasy']",5.9
4,tt0107719,The Nutcracker,"['family','fantasy','music']",5.9
5,tt0177606,Puss in Boots,"['family','fantasy','musical']",6.2
6,tt0074968,"No Deposit, No Return (No Deposit No Return)","['comedy','family']",6.4
7,tt0046387,The Sword and the Rose,"['adventure','drama','family']",6.4
8,tt0038032,Road to Utopia,"['adventure','comedy','family']",7.1
9,tt0120669,Fear and Loathing in Las Vegas,"['adventure','comedy','drama']",7.4


In [17]:
# Test 4: The Hangover
input1 = 'The Hangover'

print(f'NLP Recommendations for: {input1}')
recommend1 = get_recommendations_hybrid(input1)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating']]

NLP Recommendations for: The Hangover


,Movie ID,Title,Genre,Weighted Rating
0,tt0803057,The Promotion,['comedy'],5.6
1,tt1053810,The Big Year,['comedy'],6.2
2,tt0183649,Phone Booth,"['crime','thriller']",7.1
3,tt0166276,Monkeybone,"['comedy','fantasy']",4.8
4,tt0884732,The Wedding Ringer,"['comedy','romance']",6.6
5,tt0117108,Multiplicity,"['comedy','romance','sci-fi']",6.2
6,tt3416742,What We Do in the Shadows,"['comedy','horror']",7.6
7,tt1279935,Date Night,"['comedy','crime','romance']",6.3
8,tt0187819,Doug's 1st Movie,"['adventure','animation','comedy']",5.0
9,tt0113497,Jumanji,"['adventure','comedy','family']",7.1


# Evaluating which model is better - NLP or Genre Based?

* using a paired t-test
* using shannon entory to test uncertainty

In [18]:
import pandas as pd
from scipy import stats

# test movies
test_movies = ['The Hangover', 'Jumanji', 'Toy Story', 'Inception', 'The Dark Knight', "12 Years a Slave",\
               "Titanic", "Charlotte's Web", "Interstellar","Eternal Sunshine of the Spotless Mind", "10 Things I Hate About You", 
               "27 Dresses", "The Devil Wears Prada"]
genre_means = []
hybrid_means = []

#  average Weighted Ratings for each model across test cases
for movie in test_movies:
    # recommendations from both models
    genre_recs = genre_recommend(movie) 
    hybrid_recs = get_recommendations_hybrid(movie)
    
    # mean Weighted Rating for the top recommendations
    genre_means.append(genre_recs['Weighted Rating'].mean())
    hybrid_means.append(hybrid_recs['Weighted Rating'].mean())

# perform a paired T-Test (Parametric)
# H0: There is no difference between the models.
# H1: One model is significantly better than the other.
t_stat, p_value = stats.ttest_rel(hybrid_means, genre_means)

print(f"Hybrid Mean Rating: {sum(hybrid_means)/len(hybrid_means):.2f}")
print(f"Genre Mean Rating: {sum(genre_means)/len(genre_means):.2f}")
print(f"P-Value: {p_value:.4f}")

# Evaluation at alpha Level 0.1
alpha = 0.1
if p_value < alpha:
    print(f"Result: Significant difference found (p < {alpha}).")
    if sum(hybrid_means) > sum(genre_means):
        print("Hybrid model is statistically better.")
    else:
        print("Genre model is statistically better.")
else:
    print(f"Both models perform similarly.")

Auto-selected: Titanic (1997) — Rating: 8.0
Multiple movies found with the title 'Titanic'. Auto-selected: Titanic (1997) — Weighted Rating: 8.00
Auto-selected: Charlotte's Web (1973) — Rating: 6.9
Multiple movies found with the title 'Charlotte's Web'. Auto-selected: Charlotte's Web (1973) — Weighted Rating: 6.90
Hybrid Mean Rating: 6.63
Genre Mean Rating: 6.46
P-Value: 0.0308
Result: Significant difference found (p < 0.1).
Hybrid model is statistically better.


# What do these results mean? 

In the code, we conducted a paired t-test to check if the average difference between two related groups is significantly different from zero. Instead of looking at the two models as completely separate entities, the t-test focuses on the gap in performance for every individual movie we are test.

In [19]:
# Shannon entropy test: measurement of uncertainty

import numpy as np
from collections import Counter
import math

def calculate_entropy(genre_list):
    all_genres = [g for sublist in genre_list for g in parse_genres(sublist)]
    counts = Counter(all_genres)
    total = sum(counts.values())
    
    #  Shannon Entropy 
    probs = [count / total for count in counts.values()]
    return -sum(p * math.log2(p) for p in probs if p > 0)

genre_entropies = []
hybrid_entropies = []

for movie in test_movies:
    g_recs = genre_recommend(movie)
    h_recs = get_recommendations_hybrid(movie)
    
    genre_entropies.append(calculate_entropy(g_recs['Genre']))
    hybrid_entropies.append(calculate_entropy(h_recs['Genre']))

print(f"Genre Model Diversity (Entropy): {np.mean(genre_entropies):.4f}")
print(f"NLP Hybrid Diversity (Entropy): {np.mean(hybrid_entropies):.4f}")

Auto-selected: Titanic (1997) — Rating: 8.0
Multiple movies found with the title 'Titanic'. Auto-selected: Titanic (1997) — Weighted Rating: 8.00
Auto-selected: Charlotte's Web (1973) — Rating: 6.9
Multiple movies found with the title 'Charlotte's Web'. Auto-selected: Charlotte's Web (1973) — Weighted Rating: 6.90
Genre Model Diversity (Entropy): 1.4090
NLP Hybrid Diversity (Entropy): 2.6806


A higher score means the model is giving you a more diverse set of movies, which is often a sign of a smarter hybrid system.